In [18]:
import cv2
import mediapipe as mp
import time
import numpy as np
import math
from ctypes import cast, POINTER
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

# MediaPipe Task Setup

BaseOptions = mp.tasks.BaseOptions
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

detector = mp.tasks.vision.HandLandmarker.create_from_options(options)

# Drawing Tools

mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils

# Camera
cap = cv2.VideoCapture(0)

# VOLUM CONTROL
devices = AudioUtilities.GetSpeakers()
volume = devices.EndpointVolume

volRange = volume.GetVolumeRange()

minVol = volRange[0]
maxVol = volRange[1]

currentVol = 0
 
###

pTime = 0

while True:
    success, img = cap.read()
    
    if not success:
        continue
    
    height, width, _ = img.shape

    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_img
    )

    timestamp_ms = int(time.time() * 1000)

    results = detector.detect_for_video(
        mp_image,
        timestamp_ms
    )

    # Draw Landmarks
    if results.hand_landmarks:

        for hand_idx in range(len(results.hand_landmarks)):

            hand_landmarks = results.hand_landmarks[hand_idx]
            handedness = results.handedness[hand_idx]
            
            mp_drawing.draw_landmarks(
                img,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,

            mp_drawing.DrawingSpec(
                    color=(255,255,255),
                    thickness=2,
                    circle_radius=1
            ),

            mp_drawing.DrawingSpec(
                color=(180,180,180),
                thickness=2
            )
)            

            for landmark_idx, landmark in enumerate(hand_landmarks):

                cx = int(landmark.x * width)
                cy = int(landmark.y * height)

                cv2.circle(
                    img,
                    (cx, cy),
                    5,
                    (255, 255, 255),
                    cv2.FILLED
                )
                cv2.circle(
                    img,
                    (cx, cy),
                    7,
                    (120,120,120),
                    1
                )
            
            # DETECT RIGHT & LEFT HANDS
            hand_label = handedness[0].category_name
            
            x_coordinates = [lm.x for lm in hand_landmarks]
            y_coordinates = [lm.y for lm in hand_landmarks]

            text_x = int(min(x_coordinates) * width)
            text_y = int(min(y_coordinates) * height) - 10
    
    
            cv2.putText(
                img,
                hand_label,
                (text_x+2, text_y+2),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (90,90,90),
                2
            )
            cv2.putText(
                img,
                hand_label,
                (text_x, text_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255,255,255),
                2
            )
            
            # DETECT GESTURES
            if hand_label == "Left":
                index_up = hand_landmarks[8].y < hand_landmarks[6].y
                middle_up = hand_landmarks[12].y < hand_landmarks[10].y
                ring_up = hand_landmarks[16].y < hand_landmarks[14].y
                pinky_up = hand_landmarks[20].y < hand_landmarks[18].y
                
                gesture = "Unknown"
                
                if (
                    index_up and
                    middle_up and
                    ring_up and
                    pinky_up
                ):
                    gesture = "OPEN PALM"
                elif (
                    index_up and
                    middle_up and
                    not ring_up and
                    not pinky_up
                ):
                    gesture = "PEACE"
                    
                elif (
                    not index_up and
                    not middle_up and
                    not ring_up and
                    not pinky_up
                ):
                    gesture = "FIST"
                    
                cv2.putText(
                    img,
                    gesture,
                    (text_x, text_y - 35),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,74,76),
                    2
                )
            
            # VOLUME CONTROLLER
            if hand_label == "Right":
                thumb = hand_landmarks[4]
                index = hand_landmarks[8]
            
                x1 = int(thumb.x * width)
                y1 = int(thumb.y * height)

                x2 = int(index.x * width)
                y2 = int(index.y * height)
            
                cv2.circle(img,(x1,y1),7,(255,255,255),cv2.FILLED)
                cv2.circle(img,(x2,y2),7,(255,255,255),cv2.FILLED)
                cv2.line(img,(x1,y1),(x2,y2),(255,255,255),2)
            
                '''length = math.hypot(
                    x2 - x1,
                    y2 - y1
                )'''
                
                wrist = hand_landmarks[0]
                middle_tip = hand_landmarks[12]

                hand_size = math.hypot(
                    (middle_tip.x - wrist.x),
                    (middle_tip.y - wrist.y)
                )
                finger_distance = math.hypot(
                    (thumb.x - index.x),
                    (thumb.y - index.y)
                )
                
                if hand_size <= 0:
                    continue

                ratio = finger_distance / hand_size
                    
                vol = np.interp(
                    ratio,
                    [0.15, 1.3],
                    [minVol, maxVol]
                )
            
                volume.SetMasterVolumeLevel(
                    vol,
                    None
                )

                currentVol = np.interp(
                    ratio,
                    [0.15,1.3],
                    [0, 100]
                )
            
                cv2.putText(
                    img,
                    f"Ratio: {ratio:.2f}",
                    (10,110),
                    cv2.FONT_HERSHEY_PLAIN,
                    2,
                    (255,60,145),
                    2
                )
                cv2.putText(
                    img,
                    f"Volume: {int(currentVol)}%",
                    (10,150),
                    cv2.FONT_HERSHEY_PLAIN,
                    2,
                    (255,20,100),
                    2
                )
            
    # FPS
    cTime = time.time()

    fps = 1 / (cTime - pTime) if pTime != 0 else 0

    pTime = cTime
    
    cv2.putText(
        img,
        f"FPS: {int(fps)}",
        (10, 70),
        cv2.FONT_HERSHEY_PLAIN,
        2,
        (220,220,220),
        2
    )

    cv2.imshow("Hand Tracking", img)

    if cv2.waitKey(1) & 0xFF == 27:
        break
    
cap.release()
cv2.destroyAllWindows()

In [14]:
handedness[0]

Category(index=1, score=0.657556414604187, display_name='Left', category_name='Left')

[Category(index=0, score=0.993871808052063, display_name='Right', category_name='Right')]